# 🧑‍🤝‍🧑 The Multi-Agent Pattern from Scratch: Agents, Dependencies and a Crew

<img src="../assets/img/multiagent_pattern.png" alt="A crew of agents wired as a DAG" width="500"/>

---

Frameworks like [CrewAI](https://www.crewai.com/) and [AutoGen](https://microsoft.github.io/autogen/stable/) let
you build multi-agent applications. They implement variations of the multi-agent pattern: a task is split into
**smaller subtasks executed by different roles** (one agent writes, another translates, another files the result).

This notebook builds a **minimal version of CrewAI** around two of its concepts, **Agent** and **Crew**. It also
borrows [Airflow](https://airflow.apache.org/)'s `>>` and `<<` operators to define dependencies between agents: an
agent plays the part of an Airflow *task*, and the crew plays the part of an Airflow *DAG*.

## Learning Objectives
In this notebook, you will learn:
1. **Building blocks** - prompt helpers, a `@tool` decorator and a ReAct loop on the raw OpenAI SDK (a recap of repository 06)
2. **An `Agent` with dependencies** - a role, a task and an expected output; `>>` / `<<` pass one agent's output into another's context
3. **A `Crew`** - collect agents in a context manager, run them in topological order and print the DAG
4. **Fan-in** - `[a, b] >> c`: one agent that waits for two others and reads both outputs
5. **The bridge to LangGraph** - the same DAG as a `StateGraph`, wired with `add_edge`

## Where this fits
`01_multi_agent_vs_monolithic_agent.ipynb` showed *why* a team helps. This notebook builds the machinery a
framework like CrewAI hides — deliberately without LangChain, on the raw OpenAI SDK, so every step is visible.

## Prerequisites
- ReAct / tool calling basics (repository 06 — agent fundamentals)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

### 📜 Attribution

The prompt helpers, `Tool`, the ReAct loop, `Agent` and `Crew` below are adapted from
[neural-maze/agentic_patterns](https://github.com/neural-maze/agentic_patterns) (MIT License). Changes: shorter
docstrings, one OpenAI client for every agent, a loop that reports malformed replies back to the model instead of
crashing or spinning, context labelled with the agent that produced it, demo files written to a temporary folder,
and `Crew.run()` returning every agent's output.

<details>
<summary>MIT License</summary>

```
MIT License

Copyright (c) 2024 The Neural Maze

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```
</details>

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` from the project-root `.env` and creates a temporary folder for the files
the agents write, so a run never changes the repository.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import tempfile
import warnings
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Files the agents write land here, not next to the notebook
DEMO_DIR = Path(tempfile.mkdtemp(prefix="crew_demo_"))

print("✅ Environment ready")
print(f"   agent output folder: a fresh temporary folder named {DEMO_DIR.name}")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: one raw OpenAI client for every agent
# ============================================================================
# This notebook builds the agent machinery itself, so it calls the OpenAI SDK
# directly instead of a LangChain chat model. Same model as the rest of the course.
from openai import OpenAI

MODEL = "gpt-4o-mini"
client = OpenAI()  # reads OPENAI_API_KEY from the environment

print(f"✅ OpenAI client ready — model {MODEL}")

---
## 🧱 Part 1: Building Blocks (a Recap of Repository 06)

Every agent in this notebook is a **ReAct loop** written as plain text: the model either asks for a tool inside
`<tool_call>…</tool_call>` tags or answers inside `<response>…</response>` tags. Code runs the tool and sends the
result back as an `<observation>`. Repository 06 builds this loop step by step; this part re-creates it compactly
so the notebook runs on its own.

### 1.1 🧰 Prompt helpers
Four small functions: call the model, build a chat message, append to the history, and pull the text out of an
XML-style tag. `fancy_print` is only for readable logs.

In [ ]:
# ============================================================================
# BUILDING BLOCKS: PROMPT HELPERS
# ============================================================================
import re
import time
from dataclasses import dataclass

from colorama import Fore, Style


def completions_create(client, messages: list, model: str) -> str:
    """Send the chat history to the model and return the reply text."""
    response = client.chat.completions.create(messages=messages, model=model)
    return str(response.choices[0].message.content)


def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    """One chat message; optionally wrap the content in <tag>…</tag>."""
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


def update_chat_history(history: list, msg: str, role: str, tag: str = "") -> None:
    history.append(build_prompt_structure(prompt=msg, role=role, tag=tag))


@dataclass
class TagContentResult:
    content: list[str]  # the text inside every <tag>…</tag> found
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """Return the content of every <tag>…</tag> in the text."""
    matched = re.findall(rf"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    return TagContentResult(content=[m.strip() for m in matched], found=bool(matched))


def fancy_print(message: str) -> None:
    print(Style.BRIGHT + Fore.CYAN + f"\n{'=' * 50}")
    print(Fore.MAGENTA + message)
    print(Style.BRIGHT + Fore.CYAN + f"{'=' * 50}\n" + Style.RESET_ALL)
    time.sleep(0.5)


print(extract_tag_content("<thought>count the words</thought><response>5</response>", "response"))

### 1.2 🔧 Tools
The `@tool` decorator reads a function's name, docstring and type hints into a JSON signature. The signature is
what the model sees; `validate_arguments` casts the model's arguments to the declared types before the call.

In [ ]:
# ============================================================================
# BUILDING BLOCKS: THE TOOL ABSTRACTION
# ============================================================================
import json
from typing import Callable


def get_fn_signature(fn: Callable) -> dict:
    """Name, docstring and parameter types of a function, as a JSON-ready dict."""
    schema = {k: {"type": v.__name__} for k, v in fn.__annotations__.items() if k != "return"}
    return {"name": fn.__name__, "description": fn.__doc__, "parameters": {"properties": schema}}


def validate_arguments(tool_call: dict, tool_signature: dict) -> dict:
    """Cast each argument to its declared type (simplified: int, str, bool, float)."""
    properties = tool_signature["parameters"]["properties"]
    type_mapping = {"int": int, "str": str, "bool": bool, "float": float}
    for arg_name, arg_value in tool_call["arguments"].items():
        expected_type = properties[arg_name].get("type")  # KeyError for an unknown argument
        if not isinstance(arg_value, type_mapping[expected_type]):
            tool_call["arguments"][arg_name] = type_mapping[expected_type](https://github.com/Sourav692/ai-engineering-demystified-08-advanced-agent-systems/tree/main/modules/05_multi_agent_foundations/notebooks/arg_value)
    return tool_call


class Tool:
    """A callable plus the JSON signature the model reads."""

    def __init__(self, name: str, fn: Callable, fn_signature: str):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    def run(self, **kwargs):
        return self.fn(**kwargs)


def tool(fn: Callable) -> Tool:
    """Decorator: turn a function into a Tool."""
    fn_signature = get_fn_signature(fn)
    return Tool(name=fn_signature["name"], fn=fn, fn_signature=json.dumps(fn_signature))


@tool
def word_count(text: str) -> int:
    """Count the words in a text."""
    return len(text.split())


print("Signature the model sees:", word_count.fn_signature)
print("Direct call:", word_count.run(text="the quick brown fox jumps"))

### 1.3 🔁 The ReAct loop
`ReactAgent.run()` loops until the model answers inside `<response>` tags. Each round it runs the requested tools
and sends their results back as an `<observation>`. Two guards keep the loop honest:
- a malformed tool call (bad JSON, an unknown tool, a wrong argument) comes back to the model as an `ERROR:`
  observation instead of crashing the run;
- a reply with neither a tool call nor an answer gets an "Unrecognised step" message, so the model corrects its
  format instead of the loop silently spinning until `max_rounds`.

In [ ]:
# ============================================================================
# BUILDING BLOCKS: A REACT AGENT ON THE RAW OPENAI SDK
# ============================================================================
REACT_SYSTEM_PROMPT = """
You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools>
%s
</tools>

Example session:

<question>What's the current temperature in Madrid?</question>
<thought>I need to get the current weather in Madrid</thought>
<tool_call>{"name": "get_current_weather","arguments": {"location": "Madrid", "unit": "celsius"}, "id": 0}</tool_call>

You will be called again with this:

<observation>{0: {"temperature": 25, "unit": "celsius"}}</observation>

You then output:

<response>The current temperature in Madrid is 25 degrees Celsius</response>

Additional constraints:

- If the user asks you something unrelated to any of the tools above, answer freely enclosing your answer with <response></response> tags.
"""

UNRECOGNISED_STEP = "Unrecognised step. Reply with a <tool_call>...</tool_call> or with your final <response>...</response>."


class ReactAgent:
    """A ReAct agent whose tool calls and answers are XML-tagged text."""

    def __init__(self, tools: Tool | list[Tool], model: str = MODEL, system_prompt: str = "") -> None:
        self.client = client
        self.model = model
        self.system_prompt = system_prompt
        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {t.name: t for t in self.tools}

    def add_tool_signatures(self) -> str:
        return "".join(t.fn_signature for t in self.tools)

    def process_tool_calls(self, tool_calls_content: list) -> dict:
        """Run each requested tool; {call id: result}. A malformed call becomes an ERROR observation."""
        observations = {}
        for i, raw_call in enumerate(tool_calls_content):
            call_id = i
            try:
                tool_call = json.loads(raw_call)
                call_id = tool_call.get("id", i)
                selected = self.tools_dict[tool_call["name"]]
                tool_call = validate_arguments(tool_call, json.loads(selected.fn_signature))
                print(Fore.GREEN + f"\nUsing tool {selected.name} with {tool_call['arguments']}")
                result = selected.run(**tool_call["arguments"])
            except Exception as e:  # bad JSON, an unknown tool or a wrong argument: tell the model
                result = f"ERROR: {type(e).__name__}: {e}"
            print(Fore.GREEN + f"Tool result: {result}")
            observations[call_id] = result
        return observations

    def run(self, user_msg: str, max_rounds: int = 10) -> str:
        # Build the prompt for THIS run (appending to self.system_prompt would grow it on every call)
        system_prompt = self.system_prompt
        if self.tools:
            system_prompt += "\n" + REACT_SYSTEM_PROMPT % self.add_tool_signatures()
        chat_history = [
            build_prompt_structure(prompt=system_prompt, role="system"),
            build_prompt_structure(prompt=user_msg, role="user", tag="question"),
        ]
        if not self.tools:  # nothing to call: one plain completion
            return completions_create(self.client, chat_history, self.model)

        for _ in range(max_rounds):
            completion = completions_create(self.client, chat_history, self.model)
            response = extract_tag_content(completion, "response")
            if response.found:
                return response.content[0]

            update_chat_history(chat_history, completion, "assistant")
            thought = extract_tag_content(completion, "thought")
            if thought.found:
                print(Fore.MAGENTA + f"\nThought: {thought.content[0]}")

            tool_calls = extract_tag_content(completion, "tool_call")
            if tool_calls.found:
                observations = self.process_tool_calls(tool_calls.content)
                update_chat_history(chat_history, f"{observations}", "user", tag="observation")
            else:
                print(Fore.YELLOW + "Unrecognised step: asking for a <tool_call> or a <response>")
                update_chat_history(chat_history, UNRECOGNISED_STEP, "user")

        print(Fore.RED + f"⚠️ No <response> after {max_rounds} rounds")
        return f"FAILED: no <response> after {max_rounds} rounds"


react_agent = ReactAgent(tools=word_count)
answer = react_agent.run("How many words are in the sentence 'the quick brown fox jumps'?")
print(Style.RESET_ALL + f"\nAnswer: {answer}")
print(f"{'✅' if '5' in answer else '❌'} the answer contains the tool's count (5)")

The model asked for `word_count`, the loop ran it, and the `<observation>` came back before the final
`<response>`. The ✅ line is computed in code from the answer text.

The two guards rarely fire with a capable model, so the next cell proves them with a **scripted client**: a
stand-in for `OpenAI()` that returns canned replies — first a reply with no tags, then a call to a tool that does
not exist, then an answer. No model is called.

In [ ]:
# ============================================================================
# BUILDING BLOCKS: PROVE THE LOOP'S GUARDS WITH A SCRIPTED CLIENT (NO API CALLS)
# ============================================================================
from types import SimpleNamespace


class ScriptedClient:
    """Stands in for OpenAI(): returns canned replies in order and records every request."""

    def __init__(self, replies: list[str]):
        self.replies, self.sent = list(replies), []
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))

    def _create(self, messages, model):
        self.sent.append(list(messages))
        reply = SimpleNamespace(content=self.replies.pop(0))
        return SimpleNamespace(choices=[SimpleNamespace(message=reply)])


stub = ScriptedClient([
    "The sentence has five words.",  # no tags at all
    '<tool_call>{"name": "count_words", "arguments": {"text": "a b c d e"}, "id": 0}</tool_call>',  # unknown tool
    "<response>5</response>",
])
guarded = ReactAgent(tools=word_count)
guarded.client = stub
print(Style.RESET_ALL + "Answer:", guarded.run("How many words are in 'a b c d e'?"))

fed_back = [m["content"] for m in stub.sent[-1] if m["role"] == "user"][1:]  # skip the question
for message in fed_back:
    print("  fed back to the model:", message[:100])
ok = any(m.startswith("Unrecognised step") for m in fed_back) and any("ERROR: KeyError" in m for m in fed_back)
print(f"{'✅' if ok else '❌'} both malformed replies went back to the model, and the loop still finished")

---
## 🤝 Part 2: An `Agent` and Its Dependencies

An **`Agent`** wraps a `ReactAgent` with a *backstory* (its system prompt), a *task description*, an *expected
output* and a *context* slot. Its `run()` builds one prompt from those pieces, runs the ReAct loop, and hands the
output to every agent that depends on it.

`a >> b` means *b depends on a*: when `a` runs, its output lands in `b`'s context. `b << a` says the same thing
the other way round. Both operators return the right-hand agent, so `a >> b >> c` chains.

> **Note**: `Agent.__init__` registers each new agent with the active `Crew`. The `Crew` here is a placeholder
> that does nothing; Part 3 replaces it with the real one.

In [ ]:
# ============================================================================
# AGENT: A ROLE, A TASK, AN EXPECTED OUTPUT AND A CONTEXT
# ============================================================================
from textwrap import dedent


class Crew:
    """Placeholder until Part 3: agents created now belong to no crew."""

    current_crew = None

    @staticmethod
    def register_agent(agent):
        pass


class Agent:
    """An agent that can work in a team: dependencies in, context out."""

    def __init__(
        self,
        name: str,
        backstory: str,
        task_description: str,
        task_expected_output: str = "",
        tools: list[Tool] | Tool | None = None,
        llm: str = MODEL,
    ):
        self.name = name
        self.backstory = backstory
        self.task_description = task_description
        self.task_expected_output = task_expected_output
        self.react_agent = ReactAgent(model=llm, system_prompt=self.backstory, tools=tools or [])

        self.dependencies: list[Agent] = []  # agents this agent waits for
        self.dependents: list[Agent] = []  # agents that wait for this one
        self.context = ""

        Crew.register_agent(self)  # join the active `with Crew():` block, if any

    def __repr__(self):
        return f"{self.name}"

    def __rshift__(self, other):
        """self >> other: other depends on self."""
        self.add_dependent(other)
        return other  # allow chaining

    def __lshift__(self, other):
        """self << other: self depends on other."""
        self.add_dependency(other)
        return other  # allow chaining

    def __rrshift__(self, other):
        """[a, b] >> self: a list has no >>, so Python calls this. self depends on every item."""
        self.add_dependency(other)
        return self  # allow chaining

    def __rlshift__(self, other):
        """[a, b] << self: every item depends on self."""
        self.add_dependent(other)
        return self  # allow chaining

    def add_dependency(self, other):
        """self depends on other (an Agent or a list of Agents)."""
        others = other if isinstance(other, list) else [other]
        if not all(isinstance(item, Agent) for item in others):
            raise TypeError("The dependency must be an instance or list of Agent.")
        for item in others:
            self.dependencies.append(item)
            item.dependents.append(self)

    def add_dependent(self, other):
        """other (an Agent or a list of Agents) depends on self."""
        others = other if isinstance(other, list) else [other]
        if not all(isinstance(item, Agent) for item in others):
            raise TypeError("The dependent must be an instance or list of Agent.")
        for item in others:
            item.dependencies.append(self)
            self.dependents.append(item)

    def receive_context(self, sender: "Agent", input_data: str):
        """Store another agent's output, labelled with the agent that produced it."""
        self.context += f"Output of {sender.name}:\n{input_data}\n\n"

    def create_prompt(self) -> str:
        return dedent(
            f"""
        You are an AI agent. You are part of a team of agents working together to complete a task.
        I'm going to give you the task description enclosed in <task_description></task_description> tags. I'll also give
        you the available context from the other agents in <context></context> tags. If the context
        is not available, the <context></context> tags will be empty. You'll also receive the task
        expected output enclosed in <task_expected_output></task_expected_output> tags. With all this information
        you need to create the best possible response, always respecting the format as describe in
        <task_expected_output></task_expected_output> tags. If expected output is not available, just create
        a meaningful response to complete the task.

        <task_description>
        {self.task_description}
        </task_description>

        <task_expected_output>
        {self.task_expected_output}
        </task_expected_output>

        <context>
        {self.context}
        </context>

        Your response:
        """
        ).strip()

    def run(self) -> str:
        """Run the task, then pass the output to every dependent."""
        output = self.react_agent.run(user_msg=self.create_prompt())
        for dependent in self.dependents:
            dependent.receive_context(self, output)
        return output


print("✅ Agent defined — operators: >>, <<, and [a, b] >> c")

### 2.1 ✍️ One agent, no tools
With no tools, the agent's `ReactAgent` makes one plain completion.

In [ ]:
# ============================================================================
# AGENT: A SINGLE AGENT WITHOUT TOOLS
# ============================================================================
agent_example = Agent(
    name="Poet Agent",
    backstory="You are a well-known poet, who enjoys creating high quality poetry.",
    task_description="Write a poem about the meaning of life in less than 6 lines",
    task_expected_output="Just output the poem, without any title or introductory sentences",
)

print(agent_example.run())

### 2.2 🗂️ One agent with a tool
A tool that writes a string to a text file. The tool keeps only the file *name* the model chooses and always
writes inside the temporary folder from the setup cell.

In [ ]:
# ============================================================================
# AGENT: A TOOL THAT WRITES A TEXT FILE
# ============================================================================
@tool
def write_str_to_txt(string_data: str, txt_filename: str) -> str:
    """Write a string to a .txt file (overwriting it). Args: string_data, the text; txt_filename, the file name."""
    path = DEMO_DIR / Path(txt_filename).name  # never outside the demo folder
    path.write_text(string_data, encoding="utf-8")
    return f"Data successfully written to {path.name}"


agent_tool_example = Agent(
    name="Writer Agent",
    backstory="You are a language model specialised in writing text into .txt files",
    task_description="Write the string 'This is a Tool Agent' into a file named `tool_agent_note.txt`",
    task_expected_output="A .txt file containing the given string",
    tools=write_str_to_txt,
)

print(Style.RESET_ALL + "Final answer:", agent_tool_example.run())
written = (DEMO_DIR / "tool_agent_note.txt").read_text(encoding="utf-8") if (DEMO_DIR / "tool_agent_note.txt").exists() else None
print(f"{'✅' if written == 'This is a Tool Agent' else '❌'} file content on disk: {written!r}")

### 2.3 🔗 Defining agent dependencies
Two agents: a poet and a translator. `agent_1 >> agent_2` makes the translator depend on the poet.

In [ ]:
# ============================================================================
# DEPENDENCIES: agent_1 >> agent_2
# ============================================================================
agent_1 = Agent(
    name="Poet Agent",
    backstory="You are a well-known poet, who enjoys creating high quality poetry.",
    task_description="Write a poem about the meaning of life in 4 lines",
    task_expected_output="Just output the poem, without any title or introductory sentences",
)

agent_2 = Agent(
    name="Poem Translator Agent",
    backstory="You are an expert translator especially skilled in Ancient Greek",
    task_description="Translate a poem into Ancient Greek",
    task_expected_output="Just output the translated poem and nothing else",
)

agent_1 >> agent_2

print("Agent 1 dependencies: ", agent_1.dependencies)
print("Agent 1 dependents:   ", agent_1.dependents)
print("Agent 2 dependencies: ", agent_2.dependencies)
print("Agent 2 dependents:   ", agent_2.dependents)

Running `agent_1` adds its output to `agent_2`'s context, labelled with the agent that wrote it.

In [ ]:
# ============================================================================
# DEPENDENCIES: RUN THE FIRST AGENT, READ THE SECOND AGENT'S CONTEXT
# ============================================================================
poem = agent_1.run()
print(poem)
print("\n--- agent_2.context ---")
print(agent_2.context)
print(f"{'✅' if poem in agent_2.context else '❌'} the poem is in the translator's context")

Now the second agent runs on the context it received from the first.

In [ ]:
# ============================================================================
# DEPENDENCIES: RUN THE SECOND AGENT
# ============================================================================
print(agent_2.run())

---
## 👥 Part 3: The `Crew`

Running agents by hand in the right order doesn't scale. A **`Crew`** collects every agent created inside a
`with Crew() as crew:` block, sorts them **topologically** (an agent runs only after everything it depends on;
Kahn's algorithm, which also detects cycles), and runs them in that order. `plot()` returns the dependency graph as
a Graphviz `Digraph`.

### 3.1 🏗️ The `Crew` class

In [ ]:
# ============================================================================
# CREW: A CONTEXT MANAGER THAT RUNS AGENTS IN TOPOLOGICAL ORDER
# ============================================================================
from collections import deque

from graphviz import Digraph  # type: ignore


class Crew:
    """A group of agents run in dependency order."""

    current_crew = None  # the crew of the active `with` block

    def __init__(self):
        self.agents = []

    def __enter__(self):
        Crew.current_crew = self
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        Crew.current_crew = None

    def add_agent(self, agent):
        self.agents.append(agent)

    @staticmethod
    def register_agent(agent):
        if Crew.current_crew is not None:
            Crew.current_crew.add_agent(agent)

    def topological_sort(self) -> list:
        """Kahn's algorithm: start from agents with no dependencies; raise on a cycle."""
        in_degree = {agent: len(agent.dependencies) for agent in self.agents}
        queue = deque([agent for agent in self.agents if in_degree[agent] == 0])
        sorted_agents = []
        while queue:
            current_agent = queue.popleft()
            sorted_agents.append(current_agent)
            for dependent in current_agent.dependents:
                in_degree[dependent] -= 1
                if in_degree[dependent] == 0:
                    queue.append(dependent)
        if len(sorted_agents) != len(self.agents):
            raise ValueError("Circular dependencies detected among agents, preventing a valid topological sort")
        return sorted_agents

    def plot(self) -> Digraph:
        """The crew's DAG as a Graphviz Digraph."""
        dot = Digraph(format="png")
        for agent in self.agents:
            dot.node(agent.name)
            for dependency in agent.dependencies:
                dot.edge(dependency.name, agent.name)
        return dot

    def run(self) -> dict:
        """Run every agent in topological order; return {agent name: output}."""
        outputs = {}
        for agent in self.topological_sort():
            fancy_print(f"RUNNING AGENT: {agent}")
            outputs[agent.name] = agent.run()
            print(Fore.RED + outputs[agent.name] + Style.RESET_ALL)
        return outputs


# A cycle can't be sorted. Creating agents makes no model calls, so this check is free.
with Crew() as broken_crew:
    ping = Agent(name="Ping", backstory="-", task_description="-")
    pong = Agent(name="Pong", backstory="-", task_description="-")
    ping >> pong
    pong >> ping
try:
    broken_crew.topological_sort()
    print("❌ a cyclic crew was sorted")
except ValueError as e:
    print(f"✅ cycle detected: {e}")

### 3.2 ⛓️ A sequential crew
A poet, a translator and a writer that saves the translation with the file tool: `agent_1 >> agent_2 >> agent_3`.

In [ ]:
# ============================================================================
# CREW: POET >> TRANSLATOR >> WRITER
# ============================================================================
with Crew() as crew:
    agent_1 = Agent(
        name="Poet Agent",
        backstory="You are a well-known poet, who enjoys creating high quality poetry.",
        task_description="Write a poem about the meaning of life in less than 6 lines",
        task_expected_output="Just output the poem, without any title or introductory sentences",
    )

    agent_2 = Agent(
        name="Poem Translator Agent",
        backstory="You are an expert translator especially skilled in Spanish",
        task_description="Translate a poem into Spanish",
        task_expected_output="Just output the translated poem and nothing else",
    )

    agent_3 = Agent(
        name="Writer Agent",
        backstory="You are an expert transcriber, that loves writing poems into txt files",
        task_description="You'll receive a Spanish poem in your context. You need to write the poem into a file named `poema_es.txt`",
        task_expected_output="A txt file containing the Spanish poem received from the context",
        tools=write_str_to_txt,
    )

    agent_1 >> agent_2 >> agent_3

print("Run order:", crew.topological_sort())
print("\nDOT source of the crew's DAG:")
# crew.plot() returns a graphviz.Digraph; rendering it needs the Graphviz `dot` binary, so print its DOT source
print(crew.plot().source)

In [ ]:
# ============================================================================
# CREW: RUN AND CHECK THE FILE
# ============================================================================
outputs = crew.run()

saved = DEMO_DIR / "poema_es.txt"
on_disk = saved.read_text(encoding="utf-8") if saved.exists() else ""
same = " ".join(on_disk.split()) == " ".join(outputs["Poem Translator Agent"].split())
print(f"{'✅' if same else '❌'} the file on disk holds the translator's poem ({len(on_disk.splitlines())} lines)")

---
## 🔀 Part 4: Fan-in — `[a, b] >> c`

`Agent` also supports a **list** on the left of `>>`. A Python list has no `>>` operator, so Python calls
`c.__rrshift__([a, b])`, and `c` gains *two* dependencies. The topological sort runs `a` and `b` first (both have
no dependencies) and `c` only after both, with both outputs in its context.

In [ ]:
# ============================================================================
# FAN-IN: TWO ANALYSTS, ONE EDITOR
# ============================================================================
with Crew() as fan_in_crew:
    pros = Agent(
        name="Pros Analyst",
        backstory="You make the case for remote work.",
        task_description="List 3 benefits of remote work for software teams.",
        task_expected_output="Exactly 3 lines, one benefit per line, no title",
    )
    cons = Agent(
        name="Cons Analyst",
        backstory="You make the case against remote work.",
        task_description="List 3 drawbacks of remote work for software teams.",
        task_expected_output="Exactly 3 lines, one drawback per line, no title",
    )
    editor = Agent(
        name="Editor",
        backstory="You write balanced, concise summaries.",
        task_description="Write one balanced paragraph on remote work for software teams, using both lists in your context.",
        task_expected_output="One paragraph of 3-4 sentences, no title",
    )

    [pros, cons] >> editor

order = fan_in_crew.topological_sort()
print("Editor depends on:", editor.dependencies)
print("Run order:        ", order)
print(fan_in_crew.plot().source)
print(f"{'✅' if order[-1] is editor and set(editor.dependencies) == {pros, cons} else '❌'} "
      "the editor runs last and waits for both analysts")

In [ ]:
# ============================================================================
# FAN-IN: RUN, THEN CHECK THE EDITOR SAW BOTH OUTPUTS
# ============================================================================
fan_in_outputs = fan_in_crew.run()

for name in ("Pros Analyst", "Cons Analyst"):
    seen = fan_in_outputs[name] in editor.context
    print(f"{'✅' if seen else '❌'} the editor's context holds the {name}'s output")

Fan-in is where a team beats a pipeline: the two analysts don't depend on each other, and the editor is the one
place that needs both. In this crew the analysts still run one after the other — the sort only fixes an order.
LangGraph, next, runs independent nodes in the same step.

---
## 🌉 Part 5: The Bridge to LangGraph — `StateGraph.add_edge`

Everything the crew does has a direct LangGraph counterpart:

| From scratch | LangGraph |
|---|---|
| `Agent` | a node: a function from state to a state update |
| `a >> b` | `builder.add_edge("a", "b")` |
| `[a, b] >> c` | `builder.add_edge(["a", "b"], "c")` — `c` waits for both |
| `Agent.context` (a growing string) | typed state; a reducer merges what parallel nodes write |
| `Crew.topological_sort()` + `run()` | `compile()` + `invoke()`: nodes with no pending inputs run together in one *super-step* |
| `Crew.plot()` | `graph.get_graph()` |

The cell below rebuilds the fan-in crew as a `StateGraph` with stub nodes (no model calls), so only the wiring
is on show.

In [ ]:
# ============================================================================
# BRIDGE: THE FAN-IN CREW AS A STATEGRAPH (STUB NODES, NO MODEL CALLS)
# ============================================================================
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class CrewState(TypedDict):
    log: Annotated[list[str], operator.add]  # the reducer appends what each node returns


def make_node(name: str):
    def node(state: CrewState) -> dict:
        seen = [entry.split()[0] for entry in state["log"]]
        return {"log": [f"{name} ran (had seen: {seen or 'nothing'})"]}
    return node


builder = StateGraph(CrewState)
for name in ("pros", "cons", "editor"):
    builder.add_node(name, make_node(name))
builder.add_edge(START, "pros")
builder.add_edge(START, "cons")
builder.add_edge(["pros", "cons"], "editor")  # [pros, cons] >> editor
builder.add_edge("editor", END)
graph = builder.compile()

for edge in graph.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}")

result = graph.invoke({"log": []})
print()
for entry in result["log"]:
    print(" ", entry)
editor_runs = [e for e in result["log"] if e.startswith("editor")]
print(f"{'✅' if len(editor_runs) == 1 and result['log'][-1] == editor_runs[0] else '❌'} "
      "the editor ran once, after both analysts")

`pros` and `cons` ran in the same super-step, and neither saw the other's entry; `editor` ran once, in the next
step, with both entries in state. What LangGraph adds on top of the crew: typed state instead of a context
string, parallel execution, conditional edges and loops (a DAG cannot loop), and checkpoints.

---
## 📝 Summary

In this notebook, we learned:

### 1. Building blocks
- A `@tool` decorator that turns a function into a JSON signature, and a ReAct loop on the raw SDK that speaks in
  `<tool_call>` / `<response>` tags
- A text-protocol loop must report malformed replies back to the model, or it fails silently at `max_rounds`

### 2. An `Agent` with dependencies
- A backstory, a task description and an expected output; an agent's output becomes its dependents' context
- `agent_1 >> agent_2` makes `agent_2` depend on `agent_1`; `<<` is the reverse

### 3. A `Crew`
- A context manager that collects agents and runs them in topological order (a DAG, like Airflow tasks), and
  refuses a cycle

### 4. Fan-in
- `[a, b] >> c` gives `c` two dependencies; it runs after both and reads both outputs

### 5. The bridge to LangGraph
- `>>` is `add_edge`, `[a, b] >> c` is `add_edge(["a", "b"], "c")`; LangGraph adds typed state, parallel
  super-steps, conditional edges, loops and checkpoints
- Frameworks such as CrewAI add memory, delegation and tooling on top of the same idea

### Next Steps
- **`03_five_multi_agent_architectures.ipynb`**: the five canonical ways to wire agents together in LangGraph